# Visual exploratory analysis and feature engineering

Applied Data Science Capstone. Rodrigo Ariel Quilodran.

Historical IBM course datasets, analysis executed 7 November 2026. AI-assisted preparation; source data and limitations are disclosed.

In [1]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA, OUT, FIG = ROOT/'data', ROOT/'results', ROOT/'figures'
plt.rcParams.update({'figure.figsize': (10, 5), 'font.size': 12, 'axes.spines.top': False, 'axes.spines.right': False})


## Six visual analyses
Color distinguishes landing Class=1 from Class=0. Small orbit groups should not be treated as reliable rankings.

In [2]:
df=pd.read_csv(OUT/'dataset_part_2.csv')
colors={0:'#C6534D',1:'#187E8A'}
def scatter(x,y,name,xlabel,ylabel):
    fig,ax=plt.subplots(figsize=(10,5))
    for cls,g in df.groupby('Class'):ax.scatter(g[x],g[y],c=colors[cls],label=f'Class {cls}',alpha=.8,s=48,edgecolors='white',linewidth=.4)
    ax.set(xlabel=xlabel,ylabel=ylabel);ax.legend();ax.grid(alpha=.15);fig.tight_layout();fig.savefig(FIG/name,dpi=150);plt.show()
scatter('FlightNumber','LaunchSite','flight_site.png','Flight number','Launch site')
scatter('PayloadMass','LaunchSite','payload_site.png','Payload mass (kg)','Launch site')
orb=df.groupby('Orbit').Class.agg(['mean','count']).sort_index()
fig,ax=plt.subplots();ax.bar(orb.index,orb['mean']*100,color='#187E8A');ax.set(ylabel='Successful landings (%)',xlabel='Orbit',ylim=(0,115))
for i,r in enumerate(orb.itertuples()):ax.text(i,r.mean*100+2,f'n={r.count}',ha='center',fontsize=9)
fig.tight_layout();fig.savefig(FIG/'orbit_success.png',dpi=150);plt.show()
scatter('FlightNumber','Orbit','flight_orbit.png','Flight number','Orbit')
scatter('PayloadMass','Orbit','payload_orbit.png','Payload mass (kg)','Orbit')
yr=df.assign(Year=pd.to_datetime(df.Date).dt.year).groupby('Year').Class.agg(['mean','count'])
fig,ax=plt.subplots();ax.plot(yr.index,yr['mean']*100,'o-',color='#187E8A');ax.set(xlabel='Year',ylabel='Successful landings (%)',ylim=(-3,103));ax.grid(alpha=.15)
for year,r in yr.iterrows():ax.annotate(f'n={int(r["count"])}',(year,r['mean']*100),xytext=(0,8),textcoords='offset points',ha='center',fontsize=9)
fig.tight_layout();fig.savefig(FIG/'year_success.png',dpi=150);plt.show()

/var/folders/q4/kp0z7j1d201f9tr0p0rvcfgr0000gn/T/ipykernel_84230/1151443435.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.set(xlabel=xlabel,ylabel=ylabel);ax.legend();ax.grid(alpha=.15);fig.tight_layout();fig.savefig(FIG/name,dpi=150);plt.show()
/var/folders/q4/kp0z7j1d201f9tr0p0rvcfgr0000gn/T/ipykernel_84230/1151443435.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout();fig.savefig(FIG/'orbit_success.png',dpi=150);plt.show()


/var/folders/q4/kp0z7j1d201f9tr0p0rvcfgr0000gn/T/ipykernel_84230/1151443435.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout();fig.savefig(FIG/'year_success.png',dpi=150);plt.show()


## One-hot encoding
Encoding only the four requested columns (Orbit, LaunchSite, LandingPad and Serial) preserves the three Boolean fields and gives 80 columns. IBM also supplies a modeling matrix that expands each Boolean to two columns, giving 83. Both representations encode the same Boolean information; the model below uses the supplied 83-column convention.

In [3]:
features=['FlightNumber','PayloadMass','Orbit','LaunchSite','Flights','GridFins','Reused','Legs','LandingPad','Block','ReusedCount','Serial']
four_columns=pd.get_dummies(df[features],columns=['Orbit','LaunchSite','LandingPad','Serial']).astype(float)
print('Four-column encoding:',four_columns.shape)
assert four_columns.shape==(90,80)
four_columns.to_csv(OUT/'encoded_80.csv',index=False)
encoded=pd.get_dummies(df[features],columns=['Orbit','LaunchSite','LandingPad','Serial','GridFins','Reused','Legs']).astype(float)
reference=pd.read_csv(DATA/'dataset_part_3.csv')
assert set(encoded.columns)==set(reference.columns)
encoded=encoded[reference.columns]
assert np.allclose(encoded,reference)
encoded.to_csv(OUT/'dataset_part_3.csv',index=False)
print('Encoded feature shape:',encoded.shape)
encoded.head()

Four-column encoding: (90, 80)
Encoded feature shape: (90, 83)


,FlightNumber,PayloadMass,Flights,Block,ReusedCount,Orbit_ES-L1,Orbit_GEO,Orbit_GTO,Orbit_HEO,Orbit_ISS,...,Serial_B1058,Serial_B1059,Serial_B1060,Serial_B1062,GridFins_False,GridFins_True,Reused_False,Reused_True,Legs_False,Legs_True
0,1.0,6104.959412,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0
1,2.0,525.000000,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0
2,3.0,677.000000,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0
3,4.0,500.000000,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0
4,5.0,3170.000000,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0
